# Setup Tables for DFM PoC

Creates all required audit, metadata, and configuration tables for the ingestion pipeline.

**Run this once** before running any ingestion notebooks.

In [ ]:
# ========== Establish Workspace Context ==========
try:
    spark.sql("SELECT 1")
    print("[INFO] Workspace context initialized")
except Exception as e:
    print(f"[WARNING] Workspace context issue: {str(e)}")

print("[SETUP] Creating required tables for DFM PoC pipeline")

In [ ]:
# ========== Imports ==========
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, DoubleType

print("[INFO] Imports complete")

## 1. Run Audit Log

Tracks each ingestion run: files processed, rows ingested, errors, status.

In [ ]:
print("\n[TABLE 1] Creating run_audit_log...")

try:
    spark.sql("""
        CREATE TABLE IF NOT EXISTS run_audit_log (
            run_id STRING,
            period STRING,
            dfm_id STRING,
            files_processed INT,
            rows_ingested INT,
            parse_errors_count INT,
            drift_events_count INT,
            status STRING,
            started_at TIMESTAMP,
            completed_at TIMESTAMP,
            error_message STRING
        )
        USING DELTA
    """)
    
    # Verify table exists
    row_count = spark.table("run_audit_log").count()
    print(f"✓ run_audit_log created successfully ({row_count} existing rows)")
    
except Exception as e:
    print(f"✗ Failed to create run_audit_log: {str(e)}")

## 2. Data Quality Results

Stores validation check results from nb_validate_enhanced.ipynb

In [ ]:
print("\n[TABLE 2] Creating dq_results...")

try:
    spark.sql("""
        CREATE TABLE IF NOT EXISTS dq_results (
            run_id STRING,
            period STRING,
            check_id STRING,
            check_name STRING,
            check_type STRING,
            total_rows INT,
            failed_rows INT,
            pass_rate DOUBLE,
            status STRING,
            executed_at TIMESTAMP
        )
        USING DELTA
    """)
    
    row_count = spark.table("dq_results").count()
    print(f"✓ dq_results created successfully ({row_count} existing rows)")
    
except Exception as e:
    print(f"✗ Failed to create dq_results: {str(e)}")

## 3. Data Quality Exception Rows

Stores individual rows that failed validation checks.

In [ ]:
print("\n[TABLE 3] Creating dq_exception_rows...")

try:
    spark.sql("""
        CREATE TABLE IF NOT EXISTS dq_exception_rows (
            run_id STRING,
            period STRING,
            check_id STRING,
            source_row_id STRING,
            client_id STRING,
            isin_code STRING,
            value STRING,
            failure_reason STRING,
            detected_at TIMESTAMP
        )
        USING DELTA
    """)
    
    row_count = spark.table("dq_exception_rows").count()
    print(f"✓ dq_exception_rows created successfully ({row_count} existing rows)")
    
except Exception as e:
    print(f"✗ Failed to create dq_exception_rows: {str(e)}")

## 4. Exclusion Policies

Configuration table for Stage 2 transformation (Include/Remove logic).

In [ ]:
print("\n[TABLE 4] Creating exclusion_policies...")

try:
    spark.sql("""
        CREATE TABLE IF NOT EXISTS exclusion_policies (
            profile_id STRING,
            policy_key STRING,
            match_type STRING,
            match_value STRING,
            action STRING,
            priority INT,
            effective_from STRING,
            effective_to STRING,
            created_by STRING,
            created_at TIMESTAMP,
            notes STRING
        )
        USING DELTA
    """)
    
    row_count = spark.table("exclusion_policies").count()
    print(f"✓ exclusion_policies created successfully ({row_count} existing rows)")
    
    # Load sample policies if table is empty
    if row_count == 0:
        print("  [INFO] Loading sample exclusion policies...")
        
        spark.sql("""
            INSERT INTO exclusion_policies VALUES
            ('brown_shipley_default', 'property_fund', 'POLICY', 'property_fund', 'Remove', 3, '2025-01-01', '9999-12-31', 'system', current_timestamp(), 'Exclude Property Funds (Source sheet decision)'),
            ('brown_shipley_default', 'invalid_security', 'POLICY', 'invalid_security', 'Remove', 2, '2025-01-01', '9999-12-31', 'system', current_timestamp(), 'Exclude invalid securities (no ISIN)'),
            ('brown_shipley_default', 'manual_override', 'POLICY', 'manual_override', 'Remove', 4, '2025-01-01', '9999-12-31', 'system', current_timestamp(), 'Manual exclusions (one-off edge cases)'),
            ('brown_shipley_default', 'tpir_inclusion', 'POLICY', 'tpir_inclusion', 'Include', 10, '2025-01-01', '9999-12-31', 'system', current_timestamp(), 'Explicit TPIR inclusion')
        """)
        
        inserted_count = spark.table("exclusion_policies").count()
        print(f"  ✓ Loaded {inserted_count} sample policies")
    
except Exception as e:
    print(f"✗ Failed to create exclusion_policies: {str(e)}")

## 5. FX Rates (Optional)

Currency conversion rates for multi-currency portfolios.

In [ ]:
print("\n[TABLE 5] Creating fx_rates...")

try:
    spark.sql("""
        CREATE TABLE IF NOT EXISTS fx_rates (
            period STRING,
            from_currency STRING,
            to_currency STRING,
            rate DOUBLE,
            effective_date STRING,
            source STRING,
            updated_at TIMESTAMP
        )
        USING DELTA
    """)
    
    row_count = spark.table("fx_rates").count()
    print(f"✓ fx_rates created successfully ({row_count} existing rows)")
    
    # Load FX rates from Excel file if table is empty
    if row_count == 0:
        print("  [INFO] Loading FX rates from Excel file...")
        
        fx_file_path = "Files/config/FX Rates - Dec 2025.xlsx"
        
        try:
            # Check if file exists
            mssparkutils.fs.ls("Files/config/")
            
            # Read Excel file with pandas
            import pandas as pd
            pdf_fx = pd.read_excel(fx_file_path, dtype=str)
            
            print(f"    Loaded {len(pdf_fx)} rows from Excel")
            print(f"    Columns: {', '.join(pdf_fx.columns.tolist())}")
            
            # Convert to Spark DataFrame and normalize
            # Assuming structure: date columns as headers, currency pairs as rows
            # Adjust based on actual structure - we may need to reshape the data
            
            # For now, convert to spark and examine
            df_fx_raw = spark.createDataFrame(pdf_fx)
            df_fx_raw.show(5)
            
            print("  [INFO] Please examine the FX rates structure above")
            print("  [INFO] You may need to adjust the transformation logic based on the Excel structure")
            
            # TODO: Transform based on actual Excel structure
            # Expected output: period, from_currency, to_currency, rate, effective_date, source, updated_at
            
        except Exception as e:
            print(f"  ✗ Could not load FX rates from file: {str(e)}")
            print(f"\n  [ACTION REQUIRED]:")
            print(f"  1. Upload 'FX Rates - Dec 2025.xlsx' to your lakehouse")
            print(f"  2. Location: Files/config/")
            print(f"  3. Source: docs/Source Documents/IBM data - HC/IBM data - HC/FX Rates - Dec 2025.xlsx")
            print(f"\n  [INFO] Loading sample fallback rates for now...")
            
            # Fallback to sample rates
            spark.sql("""
                INSERT INTO fx_rates VALUES
                ('2026-03', 'USD', 'GBP', CAST(0.79 AS DOUBLE), '2026-03-01', 'manual_fallback', current_timestamp()),
                ('2026-03', 'EUR', 'GBP', CAST(0.85 AS DOUBLE), '2026-03-01', 'manual_fallback', current_timestamp()),
                ('2026-03', 'GBP', 'GBP', CAST(1.0 AS DOUBLE), '2026-03-01', 'manual_fallback', current_timestamp())
            """)
            
            print(f"  ✓ Loaded fallback sample rates")
    
except Exception as e:
    print(f"✗ Failed to create fx_rates: {str(e)}")

## Summary

In [ ]:
print("\n" + "=" * 80)
print("[SETUP COMPLETE]")
print("=" * 80)

tables = [
    "run_audit_log",
    "dq_results", 
    "dq_exception_rows",
    "exclusion_policies",
    "fx_rates"
]

print("\nVerifying all tables exist:\n")
for table_name in tables:
    try:
        count = spark.table(table_name).count()
        print(f"  ✓ {table_name:<25} ({count} rows)")
    except Exception as e:
        print(f"  ✗ {table_name:<25} (ERROR: {str(e)})")

print("\n" + "=" * 80)
print("[INFO] You can now run the ingestion notebooks")
print("=" * 80)